In [1]:
import numpy as np
from scipy.fft import dctn

from chebyshevnd.clenshaw import clenshaw_1d, clenshaw_2d, clenshaw_3d, clenshaw_4d, partial_clenshaw_4d
from chebyshevnd.utils import chebyshev_nodes,prune_trailing_slices_greedy_nd
from numba import njit
from chebyshevnd.interp4d import ChebyshevInterpolant4D

from time import (
    process_time,
    perf_counter,
)

## Parallel Definitions

In [2]:

def renormalize_chebyshev_coefficients_nd(out: np.ndarray) -> np.ndarray:
    """
    Renormalize an nD array of Chebyshev coefficients.

    Parameters
    ----------
    out : np.ndarray
        n-dimensional array of Chebyshev coefficients.

    Returns
    -------
    normalized : np.ndarray
        Renormalized coefficients with the same shape as `out`.
    """
    shape = out.shape
    ndim = out.ndim

    # Create factor arrays for each axis
    factors = []
    for n in shape:
        f = np.ones(n)
        f[0] = 2
        factors.append(f)

    # Build the scale tensor via outer products (einsum over all dims)
    einsum_subscripts = ",".join(chr(ord("a") + i) for i in range(ndim)) + "->" + "".join(chr(ord("a") + i) for i in range(ndim))
    scale = np.einsum(einsum_subscripts, *factors)

    # Normalize by scale * product of sizes
    normalized = out / (scale * np.prod(shape))

    return normalized
    


def compute_chebyshev_coefficients_nd(f_values: np.ndarray) -> np.ndarray:
    """
    Compute Chebyshev coefficients for an nD array of function values using DCT.

    Parameters
    ----------
    f_values : np.ndarray
        n-dimensional array of function values on a Chebyshev tensor grid.

    Returns
    -------
    coeffs : np.ndarray
        n-dimensional array of Chebyshev coefficients.
    """
    coeffs = dctn(f_values, type=2, norm=None)
    coeffs = renormalize_chebyshev_coefficients_nd(coeffs)
    return coeffs

## Doubled up Clenshaw Functions

In [6]:
@njit
def clenshaw_3d(coeffs, x, y, z, NzMaxes=None,
                ax=-1., bx=1., ay=-1., by=1., az=-1., bz=1.):
    """
    3D Clenshaw, reusing clenshaw_2d inside.
    """
    Nx, Ny, Nz = coeffs.shape
    if NzMaxes is None:
        NzMaxes = np.ones((Nx, Ny), dtype=np.int64) * Nz

    # Map z
    tz = (2 * z - (bz + az)) / (bz - az)

    # Evaluate along z for each (i, j)
    temp_xy = np.empty((Nx, Ny))
    for i in range(Nx):
        for j in range(Ny):
            temp_xy[i, j] = clenshaw_1d(coeffs[i, j, :], tz, NzMaxes[i, j])

    # Now evaluate the 2D interpolant in (x, y)
    return clenshaw_2d(temp_xy, x, y, ax=ax, bx=bx, ay=ay, by=by)

In [190]:
@njit
def clenshaw_4d(coeffs, x, y, z, w, NwMaxes=None,
                ax=-1., bx=1., ay=-1., by=1.,
                az=-1., bz=1., aw=-1., bw=1.):
    """
    4D Clenshaw, reusing clenshaw_3d inside.
    """
    Nx, Ny, Nz, Nw = coeffs.shape
    if NwMaxes is None:
        NwMaxes = np.ones((Nx, Ny, Nz), dtype=np.int64) * Nw

    # Map w
    tw = (2 * w - (aw + bw)) / (bw - aw)

    # Evaluate along w for each (i, j, k)
    temp_xyz = np.empty((Nx, Ny, Nz))
    for i in range(Nx):
        for j in range(Ny):
            for k in range(Nz):
                temp_xyz[i, j, k] = clenshaw_1d(coeffs[i, j, k, :], tw, NwMaxes[i, j, k])

    # Now evaluate the 3D interpolant in (x, y, z)
    return clenshaw_3d(temp_xyz, x, y, z,
                       ax=ax, bx=bx,
                       ay=ay, by=by,
                       az=az, bz=bz)

In [183]:
@njit
def clenshaw_4d(coeffs, x, y, z, w, NwMaxes=None,
                ax=-1., bx=1., ay=-1., by=1.,
                az=-1., bz=1., aw=-1., bw=1.):
    """
    Evaluate a 4D Chebyshev interpolant using Clenshaw's algorithm.

    Parameters
    ----------
    coeffs : 4D np.ndarray
        Chebyshev coefficients [Nx, Ny, Nz, Nw].
    x, y, z, w : float
        Point to evaluate.
    NwMaxes : 3D int64 array, optional
        Maximum w truncation orders for each (i, j, k).
        Shape (Nx, Ny, Nz). If None, defaults to full Nw for all.
    ax, bx, ay, by, az, bz, aw, bw : float
        Domain bounds for each coordinate.

    Returns
    -------
    float
        Value of interpolant at (x, y, z, w).
    """
    Nx, Ny, Nz, Nw = coeffs.shape
    if NwMaxes is None:
        NwMaxes = np.ones((Nx, Ny, Nz), dtype=np.int64) * Nw

    # Map to [-1,1]
    tx = (2 * x - (bx + ax)) / (bx - ax)
    ty = (2 * y - (by + ay)) / (by - ay)
    tz = (2 * z - (bz + az)) / (bz - az)
    tw = (2 * w - (aw + bw)) / (bw - aw)

    # Evaluate in w-direction first
    temp_z = np.empty((Nx, Ny, Nz))
    for i in range(Nx):
        for j in range(Ny):
            for k in range(Nz):
                temp_z[i, j, k] = clenshaw_1d(coeffs[i, j, k, :], tw, NwMaxes[i, j, k])

    # Collapse z-direction
    temp_y = np.empty((Nx, Ny))
    for i in range(Nx):
        for j in range(Ny):
            temp_y[i, j] = clenshaw_1d(temp_z[i, j, :], tz)

    # Collapse y-direction
    temp_x = np.empty(Nx)
    for i in range(Nx):
        temp_x[i] = clenshaw_1d(temp_y[i, :], ty)

    # Collapse x-direction
    return clenshaw_1d(temp_x, tx)

In [67]:
@njit
def clenshaw_4d(coeffs, x, y, z, w):
    """
    Evaluate 4D Chebyshev series using Clenshaw recurrence.

    Parameters
    ----------
    coeffs : ndarray, shape (nx, ny, nz, nw)
        Chebyshev coefficients.
    x, y, z, w : float
        Evaluation point in [-1, 1].

    Returns
    -------
    f : float
        Interpolated value.
    """
    nx, ny, nz, nw = coeffs.shape

    # Step 1: evaluate along w-axis
    tmp_w = np.zeros((nx, ny, nz))
    for i in range(nx):
        for j in range(ny):
            for k in range(nz):
                tmp_w[i, j, k] = clenshaw_1d(coeffs[i, j, k, :], w)

    # Step 2: evaluate along z-axis
    tmp_z = np.zeros((nx, ny))
    for i in range(nx):
        for j in range(ny):
            tmp_z[i, j] = clenshaw_1d(tmp_w[i, j, :], z)

    # Step 3: evaluate along y-axis
    tmp_y = np.zeros(nx)
    for i in range(nx):
        tmp_y[i] = clenshaw_1d(tmp_z[i, :], y)

    # Step 4: final evaluation along x-axis
    return clenshaw_1d(tmp_y, x)

def rescale(x, a, b):
    return 2 * (x - a) / (b - a) - 1

def evaluate(coeffs, x, y, z, ax,bx, ay, by, az, bz):
        x_scaled = rescale(x, ax, bx)
        y_scaled = rescale(y, ay, by)
        z_scaled = rescale(z, az, bz)
        return clenshaw_3d(coeffs, x_scaled, y_scaled, z_scaled)    

In [175]:
@njit
def clenshaw_1d_match_numpy(x, c):
    """
    Clenshaw for Chebyshev series sum_{k=0}^n c[k] * T_k(x),
    i.e., NumPy's chebval convention (NO 1/2 on c[0]).
    """
    n = len(c) - 1
    bkp1 = 0.0  # b_{k+1}
    bkp2 = 0.0  # b_{k+2}
    for k in range(n, 0, -1):            # k = n, n-1, ..., 1
        bk = 2.0 * x * bkp1 - bkp2 + c[k]
        bkp2, bkp1 = bkp1, bk
    return c[0] + x * bkp1 - bkp2


@njit
def clenshaw_nd(xs, coeffs):
    d = xs.shape[0]

    # Base case
    if d == 1:
        return clenshaw_1d(coeffs, xs[0])

    x_last = xs[-1]
    shape = coeffs.shape[:-1]
    tmp = np.empty(shape, dtype=np.float64)

    # Loop explicitly, no np.ndindex (Numba safer with nested loops)
    it = np.nditer(tmp, flags=['multi_index'], op_flags=['writeonly'])
    while not it.finished:
        idx = it.multi_index
        # Force 1D view along the last axis
        coeff_vec = coeffs[idx + (slice(None),)]
        it[0] = clenshaw_1d(coeff_vec, x_last)
        it.iternext()

    return clenshaw_nd(xs[:-1], tmp)

In [176]:
def prune_trailing_slices_greedy_nd(coeffs, delta=1e-12, scale='max'):
    """
    Structured pruning of nD Chebyshev coeffs by removing trailing slices
    along axes to reduce Clenshaw lengths while controlling a global relative error.

    Error control: sum(|discarded coeffs|) <= delta * S, with S given by `scale`.

    Parameters
    ----------
    coeffs : ndarray
        Chebyshev coefficients, n-dimensional.
    delta : float
        Allowed global relative error.
    scale : {'max','l1','l2', float}
        Scale S for the relative tolerance:
          - 'max' (default): S = max(abs(coeffs))
          - 'l1' : S = sum(abs(coeffs))
          - 'l2' : S = norm(coeffs.ravel(), 2)
          - float: use the provided value

    Returns
    -------
    kmax : tuple
        Truncation limits along each axis; keep coeffs[ :kmax[0], :kmax[1], ..., :kmax[n-1] ].
    discarded_abs_sum : float
        Sum of |coeffs| discarded (an upper bound on absolute uniform error).
    """
    c = np.asarray(coeffs)
    shape = c.shape
    ndim = c.ndim
    if c.size == 0:
        return (0,) * ndim, 0.0

    abs_c = np.abs(c)
    if scale == 'max':
        S = abs_c.max()
    elif scale == 'l1':
        S = abs_c.sum()
    elif scale == 'l2':
        S = np.sqrt(np.sum(abs_c**2))
    elif isinstance(scale, (int, float)):
        S = float(scale)
    else:
        raise ValueError("scale must be 'max', 'l1', 'l2', or a float")

    if S == 0.0:
        # all-zero; keep nothing
        return (0,) * ndim, 0.0

    tau = delta * S

    # Precompute L1 costs of each trailing slice along each axis
    # cost[axis][i] = L1 norm of slice at index i along that axis
    costs = []
    for ax in range(ndim):
        summed = abs_c.sum(axis=tuple(j for j in range(ndim) if j != ax))
        costs.append(summed)

    # Start from full sizes
    kmax = list(shape)
    discarded = 0.0

    while True:
        # compute cost of trimming one more slice on each axis
        next_costs = []
        for ax in range(ndim):
            idx = kmax[ax] - 1
            if idx >= 0:
                next_costs.append(costs[ax][idx])
            else:
                next_costs.append(np.inf)

        axis = int(np.argmin(next_costs))
        inc = next_costs[axis]

        if not np.isfinite(inc):
            break
        if discarded + inc > tau:
            break

        discarded += inc
        kmax[axis] -= 1

        if any(k == 0 for k in kmax):
            break

    return tuple(kmax), discarded

## Example Interpolation

In [3]:
# Example 4D function
f = lambda x, y, z, w: np.exp(x) * np.cos(y) * np.sin(z) * (1 + w**2)

# Create Chebyshev grid
nx, ny, nz, nw = 50, 50, 50, 50
ax, bx = -2. , 1.
ay, by = -1. , 2.
az, bz = -1/2. , 1.
aw, bw = -1. , 2.


x_nodes = chebyshev_nodes(nx,ax,bx)
y_nodes = chebyshev_nodes(ny,ay,by)
z_nodes = chebyshev_nodes(nz,az,bz)
w_nodes = chebyshev_nodes(nw,aw,bw)

# Sample the function
values = np.zeros((nx, ny, nz, nw))
for i, xi in enumerate(x_nodes):
    for j, yj in enumerate(y_nodes):
        for k, zk in enumerate(z_nodes):
            for l, wl in enumerate(w_nodes):
                values[i, j, k, l] = f(xi, yj, zk, wl)



In [4]:
# Compute coefficients
coeffs = compute_chebyshev_coefficients_nd(values)

interp4d = ChebyshevInterpolant4D(values, ax, bx, ay, by, az, bz, aw, bw)

In [9]:
# Evaluate interpolant
time_start = perf_counter()
interp = clenshaw_4d(coeffs, 0.2, -0.5, 0.3, 0.7, ax=ax,bx=bx, ay=ay, by=by, az=az, bz=bz, aw=aw, bw=bw )
elapsed_time = (perf_counter() - time_start)*1000  # convert to milliseconds
print(f"Elapsed time: {elapsed_time:.6f} ms")
exact = f(0.2, -0.5, 0.3, 0.7)

print(f"Interpolated value: {interp}")
print(f"Exact value: {exact}")
print(f"Difference: {abs(interp - exact)}")
print(f"Relative error: {abs(interp - exact) / abs(exact)}")

Elapsed time: 11.084375 ms
Interpolated value: 0.47197645236249974
Exact value: 0.4719764523625
Difference: 2.7755575615628914e-16
Relative error: 5.880711945839054e-16


In [10]:
time_start = perf_counter()
partial = partial_clenshaw_4d(coeffs,0.2, a=ax,b=bx)
elapsed_time = (perf_counter() - time_start)*1000  # convert to milliseconds
print(f"Elapsed time (partial_clenshaw_4d init): {elapsed_time:.6f} ms")
time_start = perf_counter()
interp = clenshaw_3d(partial, -0.5, 0.3, 0.7, ax=ay, bx=by, ay=az, by=bz, az=aw, bz=bw)
elapsed_time = (perf_counter() - time_start)*1000  # convert to milliseconds
print(f"Elapsed time for the rest of the calculation: {elapsed_time:.6f} ms")

print(f"Interpolated value: {interp}")
print(f"Exact value: {exact}")
print(f"Difference: {abs(interp - exact)}")
print(f"Relative error: {abs(interp - exact) / abs(exact)}")

Elapsed time (partial_clenshaw_4d init): 14.871750 ms
Elapsed time for the rest of the calculation: 0.387416 ms
Interpolated value: 0.4719764523624997
Exact value: 0.4719764523625
Difference: 3.3306690738754696e-16
Relative error: 7.056854335006866e-16


In [8]:
# Evaluate interpolant
time_start = perf_counter()
interp = interp4d.evaluate( 0.2, -0.5, 0.3, 0.7 )
elapsed_time = (perf_counter() - time_start)*1000  # convert to milliseconds
print(f"Elapsed time: {elapsed_time:.6f} ms")
exact = f(0.2, -0.5, 0.3, 0.7)

print(f"Interpolated value: {interp}")
print(f"Exact value: {exact}")
print(f"Difference: {abs(interp - exact)}")
print(f"Relative error: {abs(interp - exact) / abs(exact)}")

Elapsed time: 15.669958 ms
Interpolated value: 0.47197645236249974
Exact value: 0.4719764523625
Difference: 2.7755575615628914e-16
Relative error: 5.880711945839054e-16


In [11]:

kmax, discarded = prune_trailing_slices_greedy_nd(coeffs, delta=1e-6)

In [12]:
kmax, discarded

((10, 10, 8, 3), np.float64(6.020788059570576e-07))

In [22]:
interp4d.optimize(1e-8)
interp4d.effective_grid()

(12, 12, 10, 3)

In [23]:
interp = clenshaw_4d(coeffs,0.2, -0.5, 0.3, 0.7,
                        ax=ax,bx=bx, ay=ay, by=by, 
                        az=az, bz=bz, aw=aw, bw=bw )

In [24]:
time_start = perf_counter()
interp = clenshaw_4d(coeffs,0.2, -0.5, 0.3, 0.7,
                        ax=ax,bx=bx, ay=ay, by=by, 
                        az=az, bz=bz, aw=aw, bw=bw, kmax = kmax )
elapsed_time = (perf_counter() - time_start)*1000  # convert to milliseconds
print(f"Elapsed time: {elapsed_time:.6f} ms")
print(f"Interpolated value: {interp}")
print(f"Exact value: {exact}")
print(f"Difference: {abs(interp - exact)}")
print(f"Relative error: {abs(interp - exact) / abs(exact)}")

Elapsed time: 0.172167 ms
Interpolated value: 0.47197643797964317
Exact value: 0.4719764523625
Difference: 1.4382856849959325e-08
Relative error: 3.047367464619319e-08


In [25]:
time_start = perf_counter()
interp = interp4d.evaluate(0.2, -0.5, 0.3, 0.7 )
elapsed_time = (perf_counter() - time_start)*1000  # convert to milliseconds
print(f"Elapsed time: {elapsed_time:.6f} ms")
print(f"Interpolated value: {interp}")
print(f"Exact value: {exact}")
print(f"Difference: {abs(interp - exact)}")
print(f"Relative error: {abs(interp - exact) / abs(exact)}")

Elapsed time: 0.272250 ms
Interpolated value: 0.47197645237031266
Exact value: 0.4719764523625
Difference: 7.812639424287227e-12
Relative error: 1.6553027985147772e-11


In [26]:
time_start = perf_counter()
interp4d.precompute_x(0.2)
elapsed_time = (perf_counter() - time_start)*1000 # convert to milliseconds
print(f"Elapsed time for precompute: {elapsed_time:.6f} ms")

time_start = perf_counter()
interp = interp4d.evaluate_yzw(-0.5, 0.3, 0.7)
elapsed_time = (perf_counter() - time_start)*1000  # convert to milliseconds
print(f"Elapsed time: {elapsed_time:.6f} ms")
print(f"Interpolated value: {interp}")
print(f"Exact value: {exact}")
print(f"Difference: {abs(interp - exact)}")
print(f"Relative error: {abs(interp - exact) / abs(exact)}")

Elapsed time for precompute: 0.254542 ms
Elapsed time: 0.061042 ms
Interpolated value: 0.47197645239932123
Exact value: 0.4719764523625
Difference: 3.682121274550809e-11
Relative error: 7.80149360443679e-11


So using the precomputation trick, we get something for 4D which might be viable in terms of speed.